In [2]:
import warnings
warnings.filterwarnings("ignore")

In [24]:
from tqdm import tqdm
import pandas as pd
from datetime import datetime, timedelta
import sqlite3
import mysql.connector
import pyarrow

import numpy as np

In [25]:
connection = mysql.connector.connect(
    user = 'root',
    password = 'root',
    host = 'localhost',
    port = 3306,
    database = 'Historical_Data'
)
print("MySQL DB Connected")


MySQL DB Connected


In [26]:
cursor = connection.cursor()

cursor.execute("SELECT * FROM FT_HOUR_DATA")

results = cursor.fetchall()

columns = [column[0] for column in cursor.description]


df_original = pd.DataFrame(results, columns=columns)

In [27]:
df_original.sort_values(by = 'id_date', ascending = True)

,Open,High,Low,Close,adj_close,Volume,Hour,Exchange,id_exchange,id_date
0,31304,31305,31046,31070,31070,92610560,0,BTC-USD,1,20220516
23,29995,30047,29861,29879,29879,0,23,BTC-USD,1,20220516
22,30128,30165,29992,29992,29992,0,22,BTC-USD,1,20220516
21,29857,30129,29857,30079,30079,289941504,21,BTC-USD,1,20220516
20,29603,29850,29581,29850,29850,0,20,BTC-USD,1,20220516
...,...,...,...,...,...,...,...,...,...,...
17389,61511,61511,61214,61383,61383,430251008,1,BTC-USD,1,20240513
17388,61450,61754,61450,61497,61497,253396992,0,BTC-USD,1,20240513
17410,62889,62924,62728,62802,62802,149336064,22,BTC-USD,1,20240513
17398,63053,63053,62381,62683,62683,466661376,10,BTC-USD,1,20240513


In [28]:
df = df_original[['id_date', 'Hour', 'Close','Exchange']]

df['id_date'] = pd.to_datetime(df['id_date'], format='%Y%m%d')
df['datetime'] = df['id_date'] + pd.to_timedelta(df['Hour'], unit='h')

df = df[['datetime', 'Close', 'Exchange']]

In [29]:
def get_cutoff_indices(
    data: pd.DataFrame,
    n_features: int, 
    step_size:int
) -> list:
    
    stop_position = len(data) - 1
    
    subseq_first_idex = 0
    subseq_mid_idx = n_features
    subseq_last_idx = n_features + 1
    indices = []
    
    while subseq_last_idx <= stop_position:
        indices.append((subseq_first_idex, subseq_mid_idx, subseq_last_idx))
        
        subseq_first_idex += step_size
        subseq_mid_idx += step_size
        subseq_last_idx += step_size
        
    return indices

In [30]:
from tqdm import tqdm

def transform_ts_data_into_features_and_target(
    ts_data: pd.DataFrame,
    input_seq_len: int,
    step_size: int
) -> pd.DataFrame:
    """
    Slices and transposes data from time-series format into a (features, target)
    format that we can use to train Supervised ML models
    """
    assert set(ts_data.columns) == {'datetime', 'Close', 'Exchange'}

    exchanges = ts_data['Exchange'].unique()
    features = pd.DataFrame()
    targets = pd.DataFrame()
    
    for exchange in tqdm(exchanges):
        
        # keep only ts data for this `location_id`
        ts_data_one_exchange = ts_data.loc[
            ts_data.Exchange == exchange, 
            ['datetime', 'Close']
        ]

        # pre-compute cutoff indices to split dataframe rows
        indices = get_cutoff_indices(
            ts_data_one_exchange,
            input_seq_len,
            step_size
        )

        # slice and transpose data into numpy arrays for features and targets
        n_examples = len(indices)
        x = np.ndarray(shape=(n_examples, input_seq_len), dtype=np.float32)
        y = np.ndarray(shape=(n_examples), dtype=np.float32)
        hours = []
        for i, idx in enumerate(indices):
            x[i, :] = ts_data_one_exchange.iloc[idx[0]:idx[1]]['Close'].values
            y[i] = ts_data_one_exchange.iloc[idx[1]:idx[2]]['Close'].values
            hours.append(ts_data_one_exchange.iloc[idx[1]]['datetime'])

        # numpy -> pandas
        features_one_exchange = pd.DataFrame(
            x,
            columns=[f'close_previous_{i+1}_hour' for i in reversed(range(input_seq_len))]
        )
        features_one_exchange['datetime'] = datetime
        features_one_exchange['exchange'] = exchange

        # numpy -> pandas
        targets_one_exchange = pd.DataFrame(y, columns=[f'target_close_next_hour'])

        # concatenate results
        features = pd.concat([features, features_one_exchange])
        targets = pd.concat([targets, targets_one_exchange])

    features.reset_index(inplace=True, drop=True)
    targets.reset_index(inplace=True, drop=True)

    return features, targets['target_close_next_hour']

In [31]:
features, targets = transform_ts_data_into_features_and_target(
    df,
    input_seq_len=24*7*1, # one week of history
    step_size=24,
)

print(f'{features.shape=}')
print(f'{targets.shape=}')

100%|██████████| 1/1 [00:00<00:00,  3.15it/s]

features.shape=(719, 170)
targets.shape=(719,)
